# 🦕 DINO SDK - Setup de Ambiente Unity Catalog

Este notebook configura automaticamente o ambiente Unity Catalog utilizando o DINO SDK.

**Funcionalidades:**
- ✅ Criação de schema no catálogo especificado
- ✅ Criação dos volumes padrão: `_checkpoints`, `_schemas` e `raw`
- ✅ Verificação de configuração existente
- ✅ Relatório detalhado do setup

**Pré-requisitos:**
- Acesso ao Databricks Unity Catalog
- Sessão Spark ativa (disponível automaticamente no Databricks)
- Permissões para criar schemas e volumes no catálogo

## 📦 Import Required Libraries
Importando o DINO SDK e bibliotecas necessárias para o setup.

In [ ]:
# Import DINO SDK
import sys
import os

# Adicionar o caminho do dino_sdk se necessário
sys.path.append('/Workspace/Shared/dino_sdk/src')

from dino_sdk.schema_manager import SchemaManager, ensure_schema_simple
from pyspark.sql import SparkSession
import json
from datetime import datetime

print("🦕 DINO SDK - Setup de Ambiente")
print("=" * 40)
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("📦 Bibliotecas importadas com sucesso!")

## ⚙️ Initialize SDK Configuration
Configuração dos parâmetros do ambiente Unity Catalog.

In [ ]:
# =============================================================================
# CONFIGURAÇÃO DO AMBIENTE - PERSONALIZAR CONFORME NECESSÁRIO
# =============================================================================

# Parâmetros do Unity Catalog
CATALOG_NAME = "data_master_dev"  # ⚠️ ALTERAR conforme seu catálogo
SCHEMA_NAME = "bronze"            # ⚠️ ALTERAR conforme necessário
MANAGED_LOCATION = None           # Optional: especificar localização gerenciada

print("🔧 Configuração do Ambiente:")
print(f"   📚 Catálogo: {CATALOG_NAME}")
print(f"   🗂️  Schema: {SCHEMA_NAME}")
print(f"   📍 Localização Gerenciada: {MANAGED_LOCATION if MANAGED_LOCATION else 'Automática'}")

# Verificar se spark está disponível
try:
    spark_session = spark  # Disponível automaticamente no Databricks
    print(f"✅ Sessão Spark ativa: {spark_session.version}")
except NameError:
    print("❌ Sessão Spark não encontrada!")
    print("💡 Este notebook deve ser executado no ambiente Databricks")
    raise Exception("Sessão Spark não disponível")

## 🗂️ Create Schema in Specified Catalog
Criação do schema no catálogo especificado com tratamento de erros.

In [ ]:
# =============================================================================
# CRIAÇÃO DO SCHEMA
# =============================================================================

print("🗂️ CRIAÇÃO DE SCHEMA")
print("=" * 30)

# Inicializar SchemaManager
schema_manager = SchemaManager(CATALOG_NAME, SCHEMA_NAME)

# Verificar se catálogo existe
print("🔍 Verificando catálogo...")
if not schema_manager.catalog_exists(spark):
    print(f"❌ Catálogo '{CATALOG_NAME}' não existe!")
    print("💡 Verifique se o catálogo foi criado corretamente")
    print("💡 Use: CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
else:
    print(f"✅ Catálogo '{CATALOG_NAME}' encontrado")

# Criar ou verificar schema existente
print(f"\n🔍 Verificando schema {CATALOG_NAME}.{SCHEMA_NAME}...")

# Usar ensure_schema_exists para criar schema + volumes automaticamente
result = schema_manager.ensure_schema_exists(spark, MANAGED_LOCATION)

print(f"\n📊 RESULTADO DA CRIAÇÃO DO SCHEMA:")
print("=" * 40)

if result['success']:
    if result['schema_created']:
        print("✅ Schema criado com sucesso!")
    elif result['already_exists']:
        print("ℹ️ Schema já existia")
    
    if result.get('external_location'):
        print(f"📍 External Location: {result['external_location']}")
    
    print(f"📦 Volumes criados: {result.get('volumes_created', [])}")
    print(f"📦 Volumes já existentes: {result.get('volumes_existing', [])}")
    
    # Salvar resultado para próxima célula
    schema_creation_result = result
    
else:
    print("❌ Falha na criação do schema!")
    if result.get('errors'):
        for error in result['errors']:
            print(f"   ⚠️ {error}")
    
    schema_creation_result = result

## 📦 Create Default Volumes
Criação dos volumes padrão necessários para o DINO SDK.

In [ ]:
# =============================================================================
# VERIFICAÇÃO E CRIAÇÃO ADICIONAL DE VOLUMES (se necessário)
# =============================================================================

print("📦 VERIFICAÇÃO DE VOLUMES PADRÃO")
print("=" * 35)

# Volumes padrão esperados
default_volumes = ['_checkpoints', '_schemas', 'raw']

print("🔍 Volumes padrão esperados:")
for volume in default_volumes:
    print(f"   📁 {volume}")

# Verificar status atual dos volumes
print(f"\n🔍 Verificando volumes no schema {CATALOG_NAME}.{SCHEMA_NAME}...")

volume_status = {}
for volume_name in default_volumes:
    exists = schema_manager.volume_exists(spark, volume_name)
    volume_status[volume_name] = exists
    status = "✅ Existe" if exists else "❌ Não existe"
    print(f"   📦 {volume_name}: {status}")

# Criar volumes faltantes (se houver)
missing_volumes = [vol for vol, exists in volume_status.items() if not exists]

if missing_volumes:
    print(f"\n🔧 Criando volumes faltantes: {missing_volumes}")
    
    for volume_name in missing_volumes:
        print(f"\n📦 Criando volume: {volume_name}")
        result = schema_manager.create_volume(spark, volume_name)
        
        if result['success']:
            if result['volume_created']:
                print(f"✅ Volume '{volume_name}' criado com sucesso!")
            elif result['already_exists']:
                print(f"ℹ️ Volume '{volume_name}' já existia")
        else:
            print(f"❌ Falha ao criar volume '{volume_name}':")
            for error in result.get('errors', []):
                print(f"   ⚠️ {error}")
else:
    print("\n✅ Todos os volumes padrão já existem!")

## 📋 Setup Summary and Verification
Resumo final do setup e verificação da configuração.

In [ ]:
# =============================================================================
# RESUMO FINAL E VERIFICAÇÃO
# =============================================================================

print("📋 RESUMO FINAL DO SETUP")
print("=" * 30)

# Obter informações completas do schema
schema_info = schema_manager.get_schema_info(spark)

print(f"🏢 Catálogo: {schema_info['catalog']}")
print(f"🗂️  Schema: {schema_info['schema']}")
print(f"✅ Catálogo existe: {'Sim' if schema_info['catalog_exists'] else 'Não'}")
print(f"✅ Schema existe: {'Sim' if schema_info['schema_exists'] else 'Não'}")

if schema_info['external_location']:
    print(f"📍 External Location: {schema_info['external_location']}")

print(f"\n📊 Tabelas no schema: {len(schema_info['tables'])}")
if schema_info['tables']:
    for table in schema_info['tables'][:5]:  # Mostrar apenas primeiras 5
        print(f"   📄 {table}")
    if len(schema_info['tables']) > 5:
        print(f"   ... e mais {len(schema_info['tables']) - 5} tabelas")

# Verificar volumes finais
print(f"\n📦 VOLUMES DISPONÍVEIS:")
try:
    volumes_df = spark.sql(f"SHOW VOLUMES IN {CATALOG_NAME}.{SCHEMA_NAME}")
    volumes = [row.volume_name for row in volumes_df.collect()]
    
    for volume in volumes:
        is_default = "🦕" if volume in default_volumes else "📁"
        print(f"   {is_default} {volume}")
    
    print(f"\n📈 Total de volumes: {len(volumes)}")
    
    # Verificar se todos os volumes padrão estão presentes
    missing_defaults = set(default_volumes) - set(volumes)
    if missing_defaults:
        print(f"⚠️  Volumes padrão faltantes: {list(missing_defaults)}")
    else:
        print("✅ Todos os volumes padrão estão presentes!")
        
except Exception as e:
    print(f"❌ Erro ao listar volumes: {e}")

print(f"\n🎉 SETUP CONCLUÍDO!")
print("=" * 20)
print("🦕 Ambiente DINO SDK configurado e pronto para uso!")
print(f"📅 Concluído em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🧪 Test Schema and Volume Access
Teste básico de acesso aos recursos criados.

In [ ]:
# =============================================================================
# TESTE BÁSICO DE FUNCIONAMENTO
# =============================================================================

print("🧪 TESTE BÁSICO DE FUNCIONAMENTO")
print("=" * 35)

try:
    # Teste 1: Listar schemas no catálogo
    print("1️⃣ Testando acesso ao catálogo...")
    schemas_df = spark.sql(f"SHOW SCHEMAS IN {CATALOG_NAME}")
    schemas = [row.databaseName for row in schemas_df.collect()]
    print(f"   ✅ Schemas encontrados: {len(schemas)}")
    if SCHEMA_NAME in schemas:
        print(f"   ✅ Schema '{SCHEMA_NAME}' confirmado")
    
    # Teste 2: Acessar volumes
    print(f"\n2️⃣ Testando acesso aos volumes...")
    for volume_name in default_volumes:
        try:
            # Tentar listar conteúdo do volume (pode estar vazio)
            volume_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{volume_name}/"
            # Note: dbutils.fs.ls() seria ideal aqui, mas pode não estar disponível
            print(f"   ✅ Volume '{volume_name}' acessível em: {volume_path}")
        except Exception as e:
            print(f"   ⚠️  Volume '{volume_name}': {str(e)[:50]}...")
    
    # Teste 3: Verificar permissões básicas
    print(f"\n3️⃣ Testando permissões...")
    try:
        # Tentar criar uma tabela temporária simples
        test_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.dino_setup_test"
        
        spark.sql(f"""
            CREATE OR REPLACE TABLE {test_table_name}
            (setup_test STRING, created_at TIMESTAMP)
            USING DELTA
        """)
        
        # Inserir dados de teste
        spark.sql(f"""
            INSERT INTO {test_table_name} 
            VALUES ('DINO SDK Setup Test', current_timestamp())
        """)
        
        # Ler dados
        test_df = spark.sql(f"SELECT * FROM {test_table_name}")
        count = test_df.count()
        
        print(f"   ✅ Tabela de teste criada com sucesso ({count} registros)")
        
        # Limpar tabela de teste
        spark.sql(f"DROP TABLE {test_table_name}")
        print(f"   🧹 Tabela de teste removida")
        
    except Exception as e:
        print(f"   ⚠️  Erro no teste de permissões: {str(e)[:50]}...")
    
    print(f"\n🎯 TESTES CONCLUÍDOS!")
    print("✅ Ambiente está pronto para uso com DINO SDK")
    
except Exception as e:
    print(f"❌ Erro durante os testes: {e}")
    print("💡 Verifique as permissões e configurações do Unity Catalog")

## 📖 Next Steps

### Como usar o ambiente configurado:

```python
# 1. Import DINO SDK
from dino_sdk.ingestion_engine import IngestionEngine, IngestionConfig
from dino_sdk.schema_manager import SchemaManager

# 2. Configurar ingestão de dados
config = IngestionConfig(
    catalog_name=CATALOG_NAME,
    schema_name=SCHEMA_NAME,
    table_name="minha_tabela",
    source_path="/path/to/data",
    checkpoint_location=f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/_checkpoints/minha_tabela"
)

# 3. Executar ingestão
engine = IngestionEngine()
result = engine.ingest(config)
```

### Volumes criados e suas finalidades:

- **`_checkpoints`**: Armazena checkpoints para streaming e Delta Lake
- **`_schemas`**: Armazena definições de schema para validação
- **`raw`**: Volume para dados brutos antes do processamento

### Documentação completa:
- [GUIA_INGESTION_ENGINE.md](./GUIA_INGESTION_ENGINE.md)
- [Exemplos práticos](./examples/)